# Nuclear Pore Complex iMSM Tutorial
This notebook demonstrates how to run the iMSM analysis for trajectories of nucleocytoplasmic transport through the nuclear pore complex (NPC), a 50 MDa assembly comprising some 500 protein subunits called nucleopoins.

This notebook relies on some NPC-specific specialization of iMSM, which is very slightly modified from the generic pipeline such as the application of our symmetric clustering algorithm to accomodate the NPC's 8 spoked symmetry. However, these modifications are not essential to apply the framework.

Note this module has some specific input requirements (see below).

**Input data made with:**

B. Raveh, R. Eliasian, S. Rashkovits, D. Russel, R. Hayama, S. Sparks, D. Singh, R.Y.H. Lim, E. Villa, M.P. Rout, D. Cowburn, & A. Sali, Integrative mapping reveals molecular features underlying the mechanism of nucleocytoplasmic transport, Proc. Natl. Acad. Sci. U.S.A. 122 (42) e2507559122, https://doi.org/10.1073/pnas.2507559122 (2025). 

## 1. Import Required Libraries

In [ ]:
from iMSM.extensions.npc.npc import run

## 2. Notes about input data format
If you wish to used the bundled `.rmf` loaders, The format of the input data should be as following.
- Single parent folder containing all data (e.g. `simulations`)
- For each independent simulation, a numbered folder (e.g. `simulations/1/`, ..., `simulations/100/`). The simulations indexes should be supplied in the `LOAD_MD_SIMS_RANGE` parameter. 
- Each independent simulation folder should contain `.rmf` files, named according to the time point they represent and spaced out by the smallest time step. (e.g. `simulations/1/100.rmf`, ..., `simulations/1/5000.rmf` representing a 5 microsecond simulation). The timeframe should be supplied in the `LOAD_MD_START_TIME_NS` and the `LOAD_MD_END_TIME_NS` parameters.
- Within each `.rmf` file, kaps should be named kap\<radius\>. this should be supplied in the `LOAD_MD_KAP_RADIUS` parameter too. FGs should be named `["Nup2", "Nsp1", "Nup100", "Nup116", "Nup159", "Nup49", "Nup57", "Nup145", "Nup1", "Nup60"]` or any subset of this. If subset, list the nonexistant types in the `LOAD_MD_IGNORED_NUP_TYPES` parameter.

## 3. Configure and Run iMSM

In [ ]:
params = {}
checkpoint_path = ...
params['LOAD_MD_BASE_PATH'] = ...
params['LOAD_MD_KAP_SITES'] = ... # number of sites of the focal compoment kap
params['LOAD_MD_KAP_RADIUS'] = ... # radius of the focal component kap
params['LOAD_MD_KAP_AMOUNT'] = 100 # Amount of the focal kap in a single simulation
params['LOAD_MD_SIMS_RANGE'] = range(1, 31) # range of the simulations to load, e.g. range(1, 31) for 30 simulations, or [1, 3, 5] for simulations 1, 3 and 5
params['LOAD_MD_START_TIME_NS'] = 10000 # start time in ns to load the simulations, e.g. 10000 for 10 microseconds
params['LOAD_MD_END_TIME_NS'] = 30000 # end time in ns to load the simulations, e.g. 30000 for 30 microseconds
params['LOAD_MD_STEP_NS'] = 100 # step size in ns to load the simulations, e.g. 100 for loading every 100 ns (different rmf files are spaced out by this value)

params['DISTANCE_STATE_THRESHOLD_NM'] = 10 # maximum distance between two components for them to be considered interacting.
params['MICROSTATE_K'] = 5 # number of closest neighbors to consider when generating interaction histograms.
params['WINDOW_SIZE_STEPS'] = 10 # number of frames to consider when generating interaction histograms.

params['N_CLUSTERS'] = [160] # number of clusters to use for the clustering step

params['TM_PRIOR'] = 0.01 # added to all entries counts matrix

# Data subset parameters (take all data by default, but can be used to construct iMSM on only a subset of the data)
params['MODE'] = "subset"
params['DATA_SUBSET'] = 1.00
params['DATA_SUBSET_INDEX'] = 0
params['DATA_SUBSET_MODE'] = "simulation" 

params['STATE_CHOICE_METHOD'] = "distance" # For permeabilty calculation

run(checkpoint_path, params_override=params, start_stage=1, end_stage=8)